In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load
import pandas as pd
import warnings

warnings.filterwarnings('ignore')

# Use numpy to convert to arrays
import numpy as np
import seaborn as sns 
# Import tools needed for visualization
import pydot
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, BatchNormalization, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import LearningRateScheduler, EarlyStopping

%matplotlib inline
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

![](https://miro.medium.com/v2/resize:fit:2160/format:webp/1*vUKwarc7rCouMzSt0Ksakw.jpeg)

**Deep Learning for Regression** 🚀

**Introduction** 🌟

Deep learning, a thrilling subfield of machine learning, has proven its mettle across various tasks, with regression being no exception. 🎯 Regression tasks involve predicting a continuous output variable, and deep learning models shine in capturing intricate relationships in the data, making them powerful wizards for regression analysis. 🧙‍♂️

**Key Components** 🧠

**Neural Networks**
Architecture: Deep learning models often embrace neural networks with multiple layers, akin to the intricate workings of the human brain. 🧠 Common architectures include feedforward neural networks and exciting variants like convolutional neural networks (CNNs) and recurrent neural networks (RNNs). 🌐💡


**Activation Functions** ⚡

**Output Layer Activation** 🎯

For regression tasks, the output layer usually adopts the embrace of a linear activation function. This choice empowers the model to predict within a continuous range of values without imposing unnecessary constraints, offering flexibility and adaptability. 📈✨


**Loss Functions** 📉

**Mean Squared Error** (MSE) 🔍

MSE stands tall as a common and trusted companion in the realm of regression problems. 🌐 This stalwart loss function meticulously quantifies the average squared disparity between the model's predictions and the ground truth, providing a nuanced gauge of performance. 🎯💡

**Import the data**

In [ ]:
df = pd.read_csv("/kaggle/input/car-price-prediction/CarPrice_Assignment.csv")

**Check some rows of the data**

In [ ]:
df.head()

**check the column names**

In [ ]:
df.columns

**check the number of rows and columns in the dataset**

In [ ]:
df.shape

**check for null values or missing data**

In [ ]:
df.isna().sum()

**check if some rows are duplicate entry meaning the entire row is an exact copy of any other row.**

In [ ]:
df.duplicated().sum()

**run common summary statistics on the dataset**

In [ ]:
df.info()

In [ ]:
df.describe()

**check for the value counts of the unique classes present in each category column**

In [ ]:
for x in df.select_dtypes(include=['object']).columns.tolist():
    print(df[x].value_counts(),'\n\n')

In [ ]:
df.describe(include=object)

**Exploraorty Data Analysis && Visualisations**

![](https://algorit.ma/wp-content/uploads/2019/05/Exploratory-Data-Analysis.png)

**Lets study all the columns with visual representations**

**function to create the histogram plots for all the numerical columns**

In [ ]:
# create a function to visualize the numerical columns
def histogram(column):
    # Set a pleasing color palette
    sns.set_palette("viridis")

    # Create a figure and axes
    plt.figure(figsize=(10, 6))

    # Plot the histogram with KDE
    sns.histplot(data=df[column], bins=10, kde=True, color='green', edgecolor='black')

    # Add labels and title
    plt.xlabel(column)
    plt.ylabel('Frequency')
    plt.title(f'Distribution of {column}')

    # Add a grid for better readability
    plt.grid(axis='y', linestyle='--', alpha=0.7)

    # Display the plot
    plt.show()

In [ ]:
for x in df.select_dtypes(include=['int64','float64']).columns.tolist()[1:]:
    histogram(x)

**No, you do not need numerical columns with only a bell-shaped distribution for deep learning regression models. While a normal (bell-shaped) distribution has some desirable properties, such as a well-defined mean and standard deviation, deep learning models are not strictly limited to normally distributed data. Deep learning models are highly flexible and can handle a wide range of data distributions.**

**function to create count plots for all object data columns excluding car name**

In [ ]:
# create a function to visualize the categrical columns
def count_plot(column):
    # Set a pleasing color palette
    sns.set_palette("Set2")

    # Create a figure and axes
    plt.figure(figsize=(12, 8))

    # Plot the count plot with adjusted bar width and edge color
    sns.countplot(data=df, x=column, order=df[column].value_counts().index, palette="viridis",
                  edgecolor='black', linewidth=1.2)

    # Add labels and title with increased font size
    plt.title(f'Count Plot - {column}', fontsize=18)
    plt.xlabel(column, fontsize=14)
    plt.ylabel('Count', fontsize=14)

    # Rotate x-axis labels and adjust font size for better readability
    plt.xticks(rotation=45, ha='right', fontsize=12)

    # Add a horizontal grid for better readability
    plt.grid(axis='y', linestyle='--', alpha=0.7)

    # Display the plot
    plt.show()

In [ ]:
for x in df.select_dtypes(include=['object']).columns.tolist()[1:]:
    count_plot(x)

**Our target variable is PRICE so we will visualise the relation of various features with the price column.**

In [ ]:

def price_box_plot(column):
    # Set a pleasing color palette
    sns.set_palette("pastel")

    # Create a figure and axes
    plt.figure(figsize=(12, 8))

    # Plot the box plot with adjusted box width and whisker length
    sns.boxplot(x=column, y='price', data=df, width=0.5, fliersize=5, palette="Set3")

    # Add labels and title with increased font size
    plt.title(f'Box Plot: {column} vs. Price', fontsize=18)
    plt.xlabel(column, fontsize=14)
    plt.ylabel('Price', fontsize=14)

    # Rotate x-axis labels and adjust font size for better readability
    plt.xticks(rotation=45, ha='right', fontsize=12)

    # Add a horizontal grid for better readability
    plt.grid(axis='y', linestyle='--', alpha=0.7)

    # Display the plot
    plt.show()

# Create box plots for each categorical column
for column in df.select_dtypes(include=['object']).columns.tolist()[1:]:
    price_box_plot(column)


**data engineering on Car Name column**

![](https://k21academy.com/wp-content/uploads/2022/04/Napa-Data-Engineering-Image.jpg)

In [ ]:
# see the unique values
df['CarName'].unique()

In [ ]:
# create a function to fix the CarName column
def clean_car_name(car_name):
    return car_name.split(" ")[0].lower()
# apply the clean_car_name function to the CarName column
df['CarName'] = df['CarName'].apply(clean_car_name)
# see the unique values after fixing
df['CarName'].unique()

In [ ]:
# Fix typing mistakes
df['CarName'] = df['CarName'].str.replace('vw', 'volkswagen')
df['CarName'] = df['CarName'].str.replace('vokswagen', 'volkswagen')
df['CarName'] = df['CarName'].str.replace('toyouta', 'toyota')
df['CarName'] = df['CarName'].str.replace('maxda', 'mazda')
df['CarName'] = df['CarName'].str.replace('porcshce', 'porsche')

In [ ]:
# see the unique values after fixing typing mistakes
df['CarName'].unique()

In [ ]:
count_plot('CarName')

In [ ]:
correlation_matrix = df[df.select_dtypes(include=['int64','float64']).columns[1:]].corr()
plt.figure(figsize=(12,12))
sns.heatmap(correlation_matrix, annot=True, cmap='cool')
plt.title("Correlation Heatmap")
plt.show()


In [ ]:
df.head()

In [ ]:
df.columns

In [ ]:
df.drop('car_ID',axis=1,inplace=True)

In [ ]:
df.head()

In [ ]:
categorical_columns=df.select_dtypes(include=['object']).columns.tolist()
print(categorical_columns)

In [ ]:
numerical_columns=df.select_dtypes(include=['int64','float64']).columns.tolist()
numerical_columns=numerical_columns[:-1]
print(numerical_columns)

**Get X and Y from the dataset**

In [ ]:
X=df.iloc[:, :-1]
y=df['price']

In [ ]:
X.head()

In [ ]:
y.head()

![](https://daxg39y63pxwu.cloudfront.net/images/blog/data-preprocessing-techniques-and-steps/image_13091084341635516423259.png)

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# X contains features, y contains target variable
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.25, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

# Create transformers for categorical and numerical columns
categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(sparse=False, handle_unknown='ignore', drop='first'))
])

numerical_transformer = Pipeline(steps=[
    ('scaler', MinMaxScaler())  # Change StandardScaler to MinMaxScaler
])

# Combine transformers using ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', categorical_transformer, categorical_columns),
        ('num', numerical_transformer, numerical_columns)
    ], 
    remainder='passthrough'  # Include non-transformed columns
)

# Apply transformations to training, validation, and test sets
X_train_preprocessed = preprocessor.fit_transform(X_train)
X_val_preprocessed = preprocessor.transform(X_val)
X_test_preprocessed = preprocessor.transform(X_test)

# Get the column names after transformation
transformed_feature_names = preprocessor.get_feature_names_out()
column_names_after_transform = transformed_feature_names.tolist() + X.columns.difference(categorical_columns + numerical_columns).tolist()

# Check the preprocessed DataFrames
print(pd.DataFrame(X_train_preprocessed, columns=column_names_after_transform).head())
print(pd.DataFrame(X_val_preprocessed, columns=column_names_after_transform).head())
print(pd.DataFrame(X_test_preprocessed, columns=column_names_after_transform).head())


In [ ]:
X_train_preprocessed=pd.DataFrame(X_train_preprocessed, columns=column_names_after_transform)

In [ ]:
X_val_preprocessed=pd.DataFrame(X_val_preprocessed, columns=column_names_after_transform)

In [ ]:
X_test_preprocessed=pd.DataFrame(X_test_preprocessed, columns=column_names_after_transform)

In [ ]:
X_train_preprocessed.head()

In [ ]:
#calculate the average price of the train dataset
mean_price = y_train.mean()
print("Average price :",mean_price)

In [ ]:
#Calculate the Mean Absolute Error on the test dataset
print("MAE for Test Data:",abs(y_test - mean_price).mean())

![](https://i.ytimg.com/vi/IzcX9bTJLj8/maxresdefault.jpg)

In [ ]:
from keras.models import Sequential
from keras.layers import Dense, Dropout

🚀 **Designing the DNN - Step-by-Step Guide** 🚀

1. **Start Small 🌱:**
   - Begin with a modest architecture, perhaps with two layers.
   - Train the model and evaluate its performance on your data.
   - If it works well, you might have found a suitable model!

2. **Check for Success 🎉:**
   - If the smaller architecture succeeds, congratulations!
   - Your model might be capturing the essential patterns in the data.

3. **When Small Fails ☹️ - Go Bigger! 🚀:**
   - If the smaller model struggles or underperforms, consider increasing the size.
   - Add more layers or neurons to each layer.
   - This can enhance the model's capacity to learn complex patterns.

4. **Check Larger Networks 🏗️:**
   - Evaluate the performance of the larger model with two layers.
   - Sometimes, a bit more complexity is all you need.

5. **When Larger Two-Layers Fail 😞 - Go Deeper! 🔄:**
   - If the larger two-layer network still falls short, try going deeper.
   - Add more layers to the architecture.

6. **Check Larger and Deeper Networks 🏰:**
   - Evaluate the performance of the larger and deeper network.
   - Going deeper allows the model to capture intricate relationships.

7. **When Larger and Deeper Fail 😢 - Go Even Larger and Even Deeper! 🚀🔄:**
   - If everything else fails, be bold!
   - Increase both the size (more layers or neurons) and depth of the network.

8. **Check the Mighty Architecture 🚀🏰:**
   - Train and evaluate the mighty architecture you've created.
   - This approach might capture the most intricate patterns in your data.

9. **When All Else Fails 😫 - Revisit the Data 🔄:**
   - If, despite all efforts, the model performance is unsatisfactory, revisit your data.
   - Ensure your features are informative and the target variable is well-defined.
   - Consider collecting more data if possible.

10. **Optimize and Iterate 🔄🚀:**
    - Fine-tune hyperparameters and experiment with different architectures.
    - Iterate through these steps until you find the right balance between model complexity and data fit.

Remember, the key is to adapt and iterate based on the performance at each step. Happy modeling! 🚀🤖

In [ ]:
input_dim = X_train_preprocessed.shape[1]

model = Sequential()
model.add(Dense(150, input_dim=input_dim, activation="relu"))
model.add(Dense(1, activation="linear"))

In [ ]:
#Configure the model
model.compile(optimizer='adam',loss="mean_absolute_error", metrics=["mean_absolute_error"])

In [ ]:
#Train the model
history1=model.fit(X_train_preprocessed.values,y_train.values, validation_data=(X_val_preprocessed,y_val),epochs=10,batch_size=64)

In [ ]:
#Use the model's evaluate method to predict and evaluate the test datasets
result = model.evaluate(X_test_preprocessed.values,y_test.values)

print(model.metrics_names)


In [ ]:
#Print the results
for i in range(len(model.metrics_names)):
    print("Metric ",model.metrics_names[i],":",str(round(result[i],2)))

In [ ]:
# Plot training and validation loss
plt.plot(history1.history['loss'])
plt.plot(history1.history['val_loss'])
plt.title("Model's Training & Validation loss across epochs")
plt.ylabel('Loss')
plt.xlabel('Epochs')
plt.legend(['Train', 'Validation'], loc='upper right')
plt.show()

🔄 **Elevating the Model** 🚀

In the upgraded network, we've introduced two additional layers, each mirroring the neuron count of its predecessor. 🧠✨

To fine-tune our learning, we're making a switch in our loss function from Mean Absolute Error (MAE) to the more nuanced Mean Squared Error (MSE). This shift allows us to capture subtler variations and refines the model's ability to discern patterns within the data. 📊🔍

As we embark on this journey of enhancement, let's embrace the iterative nature of model improvement, seeking a delicate balance between complexity and performance. 🤖🛠️ Let the training begin! 🚀🔥

In [ ]:
# Define a learning rate scheduler
def lr_scheduler(epoch, lr):
    if epoch < 5:
        return lr
    else:
        return lr * 0.95  # Adjust the decay factor as needed

# Create the model
model = Sequential()
model.add(Dense(256, input_dim=input_dim, activation="relu"))
model.add(BatchNormalization())
model.add(Dropout(0.2))
model.add(Dense(256, activation="relu"))
model.add(BatchNormalization())
model.add(Dropout(0.2))
model.add(Dense(128, activation="relu"))
model.add(BatchNormalization())
model.add(Dropout(0.2))
model.add(Dense(1, activation="linear"))

# Compile the model with a lower learning rate
model.compile(optimizer=Adam(learning_rate=0.001), loss="mean_squared_error", metrics=["mean_absolute_error"])

# Train the model with the learning rate scheduler
history = model.fit(X_train_preprocessed.values, y_train.values, 
                    validation_data=(X_val_preprocessed, y_val),
                    epochs=20, batch_size=64, callbacks=[LearningRateScheduler(lr_scheduler)])

# Evaluate the model on the test set
result = model.evaluate(X_test_preprocessed, y_test)

# Print evaluation metrics
for i in range(len(model.metrics_names)):
    print("Metric", model.metrics_names[i], ":", round(result[i], 2))

In [ ]:
# Create a more complex model
model = Sequential()
model.add(Dense(512, input_dim=input_dim, activation="relu"))
model.add(BatchNormalization())
model.add(Dropout(0.5))
model.add(Dense(256, activation="relu"))
model.add(BatchNormalization())
model.add(Dropout(0.5))
model.add(Dense(128, activation="relu"))
model.add(BatchNormalization())
model.add(Dropout(0.5))
model.add(Dense(1, activation="linear"))

# Compile the model with a lower learning rate
model.compile(optimizer=Adam(learning_rate=0.001), loss="mean_squared_error", metrics=["mean_absolute_error"])

# Define early stopping to stop training if the validation loss doesn't improve
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# Train the model with the learning rate scheduler and early stopping
history = model.fit(X_train_preprocessed.values, y_train.values, 
                    validation_data=(X_val_preprocessed, y_val),
                    epochs=50, batch_size=64, 
                    callbacks=[LearningRateScheduler(lr_scheduler), early_stopping])

# Evaluate the model on the test set
result = model.evaluate(X_test_preprocessed.values, y_test)

# Print evaluation metrics
for i in range(len(model.metrics_names)):
    print("Metric", model.metrics_names[i], ":", round(result[i], 2))

In [ ]:
# Create a more complex model
model = Sequential()
model.add(Dense(512, input_dim=X_train_preprocessed.shape[1], activation="relu"))
model.add(BatchNormalization())
model.add(Dropout(0.5))
model.add(Dense(256, activation="relu"))
model.add(BatchNormalization())
model.add(Dropout(0.5))
model.add(Dense(128, activation="relu"))
model.add(BatchNormalization())
model.add(Dropout(0.5))
model.add(Dense(1, activation="linear"))

# Compile the model with a lower learning rate and L2 regularization
model.compile(optimizer=Adam(learning_rate=0.0001), 
              loss="mean_squared_error", 
              metrics=["mean_absolute_error"])

# Define early stopping to stop training if the validation loss doesn't improve
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# Train the model with the learning rate scheduler and early stopping
history = model.fit(X_train_preprocessed, y_train, 
                    validation_data=(X_val_preprocessed, y_val),
                    epochs=100, batch_size=64, 
                    callbacks=[early_stopping])

# Evaluate the model on the test set
result = model.evaluate(X_test_preprocessed, y_test)

# Print evaluation metrics
for i in range(len(model.metrics_names)):
    print("Metric", model.metrics_names[i], ":", round(result[i], 2))

As we can see that we are not able to go any further even if increasing the complexity of the model.

In [ ]:
import matplotlib.pyplot as plt

# Assuming 'history' is available after training the model

# Plot training and validation loss
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title("Model's Training & Validation loss across epochs")
plt.ylabel('Loss')
plt.xlabel('Epochs')
plt.legend(['Train', 'Validation'], loc='upper right')
plt.show()


In [ ]:
from tensorflow.keras.regularizers import l2

# Create a more complex model with L2 regularization
model = Sequential()
model.add(Dense(512, input_dim=X_train_preprocessed.shape[1], activation="relu", kernel_regularizer=l2(0.01)))
model.add(BatchNormalization())
model.add(Dropout(0.5))
model.add(Dense(256, activation="relu", kernel_regularizer=l2(0.01)))
model.add(BatchNormalization())
model.add(Dropout(0.5))
model.add(Dense(128, activation="relu", kernel_regularizer=l2(0.01)))
model.add(BatchNormalization())
model.add(Dropout(0.5))
model.add(Dense(1, activation="linear"))

# Compile the model with a lower learning rate
model.compile(optimizer=Adam(learning_rate=0.0001), 
              loss="mean_squared_error", 
              metrics=["mean_absolute_error"])

# Define early stopping to stop training if the validation loss doesn't improve
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# Train the model with the learning rate scheduler and early stopping
history = model.fit(X_train_preprocessed, y_train, 
                    validation_data=(X_val_preprocessed, y_val),
                    epochs=100, batch_size=64, 
                    callbacks=[early_stopping])

# Evaluate the model on the test set
result = model.evaluate(X_test_preprocessed, y_test)

# Print evaluation metrics
for i in range(len(model.metrics_names)):
    print("Metric", model.metrics_names[i], ":", round(result[i], 2))


In [ ]:
from tensorflow.keras.optimizers.schedules import ExponentialDecay

# Increase model complexity
model = Sequential()
model.add(Dense(512, input_dim=X_train_preprocessed.shape[1], activation="relu"))
model.add(Dense(256, activation="relu"))
model.add(Dense(128, activation="relu"))
model.add(Dense(1, activation="linear"))

# Implement a learning rate schedule
initial_learning_rate = 0.001
lr_schedule = ExponentialDecay(
    initial_learning_rate, decay_steps=10000, decay_rate=0.9, staircase=True
)
optimizer = Adam(learning_rate=lr_schedule)

# Compile the model with the learning rate schedule
model.compile(optimizer=optimizer, 
              loss="mean_squared_error", 
              metrics=["mean_absolute_error"])

# Train the model with early stopping
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
history = model.fit(X_train_preprocessed, y_train, 
                    validation_data=(X_val_preprocessed, y_val),
                    epochs=100, batch_size=64, 
                    callbacks=[early_stopping])

# Evaluate the model on the test set
result = model.evaluate(X_test_preprocessed, y_test)

# Print evaluation metrics
for i in range(len(model.metrics_names)):
    print("Metric", model.metrics_names[i], ":", round(result[i], 2))


In [ ]:
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title("Model's Training & Validation loss across epochs")
plt.ylabel('Loss')
plt.xlabel('Epochs')
plt.legend(['Train', 'Validation'], loc='upper right')
plt.show()

This is a very good imporvement in the mean absolute error score which also is near to the random forest model score in the next steps which i used for benchmarking and comparison.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Create a RandomForestRegressor for comparison
rf_model = RandomForestRegressor()
rf_model.fit(X_train_preprocessed, y_train)

# Evaluate RandomForestRegressor on the test set
rf_predictions = rf_model.predict(X_test_preprocessed)
rf_mae = mean_absolute_error(y_test, rf_predictions)
rf_mse = mean_squared_error(y_test, rf_predictions)

print("RandomForestRegressor Metrics:")
print("MAE:", round(rf_mae, 2))
print("MSE:", round(rf_mse, 2))

# Evaluate the Neural Network on the test set
nn_result = model.evaluate(X_test_preprocessed, y_test)
for i in range(len(model.metrics_names)):
    print("Metric", model.metrics_names[i], ":", round(nn_result[i], 2))


Let us also create an ensmeble and get the metric scores to see if it improves our mean absolute error metric

In [ ]:
# Ensemble prediction using average of Neural Network and Random Forest
nn_predictions = model.predict(X_test_preprocessed).flatten()  # Flatten to 1D
ensemble_predictions = (nn_predictions + rf_predictions) / 2

ensemble_mae = mean_absolute_error(y_test, ensemble_predictions)
ensemble_mse = mean_squared_error(y_test, ensemble_predictions)

print("Ensemble Metrics:")
print("MAE:", round(ensemble_mae, 2))
print("MSE:", round(ensemble_mse, 2))


In [ ]:
plt.plot(y_test, label='True Labels', marker='o')
plt.plot(ensemble_predictions, label='Ensemble Predictions', marker='o')

plt.title('True Labels vs. Ensemble Predictions')
plt.xlabel('Data Points')
plt.ylabel('Values')
plt.legend()
plt.show()

![](https://img.huffingtonpost.com/asset/585ace611c000011070ecde6.jpeg?cache=XjoiCQ33UR&ops=scalefit_720_noupscale)

![](https://media.makeameme.org/created/please-upvote-and.jpg)